# Phase 15: External Validation: Localization-Trained Model on COACH420/HOLO4K

Phase 14 showed the auxiliary localization loss beats the best prior method
(+0.0478 DCC, "aux-loss model + own centroid" vs "baseline + Phase 13's tuned
heuristic") on our own held-out split, with negligible cost to classification
(-0.0026 PR-AUC). That result earned this step: retrain the same architecture
as a deployment ensemble on all 2,056 training chains, then evaluate DCC/DCA on
the real external benchmark, COACH420 and HOLO4K, leakage-checked against our
training set exactly as in Phase 12, using the model's own trained centroid
output directly. No clustering heuristic anywhere in this notebook.

**Reference points, both already independently verified:**

| | COACH420 (n≈174) | HOLO4K (n≈967) |
|---|---|---|
| Phase 12 model (classification-only + Phase-12-heuristic) DCC | 0.368 | 0.403 |
| P2Rank, same-run, same clean subset | 0.471 | 0.551 |

Phase 13's tuned heuristic improved the Phase-12 model to a *pooled* (not
per-benchmark) DCC of 0.435 on its own tune/test split of this same clean set.
This notebook's number is directly comparable to the Phase 12 row above (same
benchmark, same leakage filter, same scoring code) and informative against
P2Rank, with the same caveat as Phase 12: P2Rank is re-run by us on the same
structures (controlled); nothing here claims to beat literature EquiPocket
numbers, for the reason established in Phase 12 (the clean subset is
measurably easier than the full published benchmark).

**For multi-chain HOLO4K structures**, each chain's summed classification
probability is used as a per-chain confidence score, and the chain with the
highest confidence contributes its own-centroid prediction for that structure
- the same "most likely to contain the pocket" logic as Phase 12/13, just
applied to pick among trained centroids instead of clustered ones.

## Cell 1: Setup

In [ ]:
!pip install -q torch-geometric==2.6.1 gudhi==3.10.1 biopython==1.84 transformers==4.46.3
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y default-jre-headless > /dev/null 2>&1

In [ ]:
import csv as _csv
import gzip, json, os, shutil, subprocess, time, urllib.request, warnings
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore")

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob10 = torch.load(locate("phase10_graphs.pt"), map_location="cpu", weights_only=False)
blob6b = torch.load(locate("phase6b_graphs.pt"), map_location="cpu", weights_only=False)
train_dataset = blob10["dataset"]
N_PRE = int(blob10["n_pre_topo"]); TOPO_DIM = int(blob10["topo_dim"])
POS_WEIGHT = float(blob10["pos_weight"])
TOPO_NAMES = blob6b["topo_names"]

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")
print(f"training set: {len(train_dataset):,} chains, node dim {train_dataset[0].x.size(1)}")

WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

# Phase12's setup cell (reused above) doesn't import these -- Phase14's reused
# model/training code (below) needs them. Add explicitly rather than relying on
# import order across two differently-assembled notebooks.
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score

# Phase12's setup names this train_dataset; Phase14's reused true-center-recovery
# code (next cell) expects `dataset` -- alias so both names resolve consistently.
dataset = train_dataset

## Cell 2: Recover true ligand centers for the TRAINING set (Phase 14's logic)

Needed to train the auxiliary loss. This re-fetches and re-parses our own
2,056 training chains' structures, a separate PDB-ID set from the benchmark,
fetched and cached independently.

In [ ]:
from Bio.PDB import PDBParser

AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}
NON_LIGAND={"HOH","DOD","WAT","SO4","PO4","GOL","EDO","PEG","PG4","PGE","1PE","MPD","TRS",
            "ACT","ACY","FMT","CIT","MES","EPE","DMS","IMD","BME","NO3","AZI","CO3","IOD",
            "BR","CL","F","NA","K","CS","RB","LI","UNX","UNL","UNK"}
CATALYTIC_METALS={"ZN","MG","MN","FE","FE2","CU","CU1","CA","NI","CO","CD","W","MO"}
MIN_LIG_ATOMS = 6
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

need = sorted({d.pdb_id for d in dataset})
def fetch(pid):
    for ext in ("pdb","cif"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500: return pid, p
    for ext in ("pdb","cif"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                b = r.read()
            if len(b) > 500: p = RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid, p
        except Exception: continue
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=16) as ex: got = list(ex.map(fetch, need))
paths = {p: f for p, f in got if f}
print(f"fetched {len(paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

def ligands_and_ca(pid, path, chain_id, n_expect):
    from Bio.PDB import MMCIFParser
    try:
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with gzip.open(path,"rt") as fh:
            model = next(parser.get_structure(pid, fh).get_models())
        ch = model[chain_id]
    except Exception:
        return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
    if len(residues) != n_expect: return None
    raw_ca = np.asarray([r["CA"].coord for r in residues], dtype=np.float32)

    ligs = []
    for c2 in model:
        for res in c2:
            if res.id[0] == " ": continue
            nm = res.get_resname().strip().upper()
            if nm in AA3_TO_1 or nm in NON_LIGAND: continue
            xyz = np.asarray([a.coord for a in res.get_unpacked_list() if a.element not in ("H","D")],
                             dtype=np.float32).reshape(-1,3)
            is_metal = nm in CATALYTIC_METALS
            if is_metal or len(xyz) >= MIN_LIG_ATOMS: ligs.append(xyz)
    return raw_ca, ligs

t0=time.time(); n_ok=0; n_geom_fail=0; n_no_ligand=0
for d in dataset:
    if d.pdb_id not in paths: continue
    r = ligands_and_ca(d.pdb_id, paths[d.pdb_id], d.chain_id, int(d.num_nodes))
    if r is None: continue
    raw_ca, ligs = r
    if not ligs: n_no_ligand += 1; continue

    # geometry-drift check: pairwise CA distances must match d.pos (residue ORDER, not just count)
    k = min(50, len(raw_ca))
    a = np.linalg.norm(raw_ca[:k,None]-raw_ca[None,:k], axis=-1)
    b = np.linalg.norm(d.pos.numpy()[:k,None]-d.pos.numpy()[None,:k], axis=-1)
    if float(np.abs(a-b).max()) > 1e-2:
        n_geom_fail += 1; continue

    offset = raw_ca.mean(axis=0)                       # same centering d.pos already used
    y_np = d.y.numpy().astype(bool)
    contact_centroid = raw_ca[y_np].mean(axis=0) if y_np.any() else raw_ca.mean(axis=0)
    # "primary" ligand = the one whose center of mass is nearest the labelled contact residues
    coms = [lig.mean(axis=0) for lig in ligs]
    best = int(np.argmin([np.linalg.norm(c - contact_centroid) for c in coms]))
    d.true_center = torch.from_numpy((coms[best] - offset).astype(np.float32))
    n_ok += 1

print(f"\ntrue centers recovered: {n_ok:,}/{len(dataset):,} "
      f"({(time.time()-t0)/60:.1f} min) | geometry mismatches: {n_geom_fail} | no ligand: {n_no_ligand}")

dataset = [d for d in dataset if hasattr(d, "true_center")]
train_dataset = dataset   # keep both names in sync for cells reused from Phase 12
n_orig = len(blob10["dataset"])
print(f"training chains with recovered true centers: {len(dataset):,}/{n_orig:,}")

# Deployment split: no held-out test needed here (the external benchmark IS
# the test) -- small internal val slice purely for early stopping, matching
# the Phase 12 deployment-training pattern.
rng = np.random.RandomState(42)
perm = rng.permutation(len(dataset))
n_val = max(1, int(0.08 * len(perm)))
splits = {"val": [dataset[i] for i in perm[:n_val]], "train": [dataset[i] for i in perm[n_val:]]}
n_tr, n_va = len(splits["train"]), len(splits["val"])
print(f"deployment training: {n_tr} train / {n_va} val")

## Cell 3: Model (identical to Phase 14)

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    """Same TopologicalEGNN as every prior phase, plus a second head that
    predicts one pocket-center point per chain via a batch-aware softmax over
    a dedicated per-residue localization logit -- decoupled from the
    classification probability on purpose (see markdown above)."""
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)                    # sums to 1 per graph
            weighted = attn.unsqueeze(-1) * pos                     # (N,3)
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

## Cell 4: Equivariance check (re-verified on this run, same discipline as always)

In [ ]:
@torch.no_grad()
def check_center_equivariance(n_trials=8):
    model = LocalizingEGNN(N_PRE, TOPO_DIM).to(DEVICE).eval()
    probe = dataset[0].clone()
    probe.batch = torch.zeros(probe.num_nodes, dtype=torch.long)
    probe = probe.to(DEVICE)
    _, c0 = model(probe)
    worst = 0.0
    for s in range(n_trials):
        g = torch.Generator().manual_seed(s)
        Q,_ = torch.linalg.qr(torch.randn(3,3,generator=g)); Q = (Q*torch.sign(torch.det(Q))).to(DEVICE)
        t = (torch.randn(3, generator=g) * 25.0).to(DEVICE)
        p2 = probe.clone(); p2.pos = probe.pos @ Q.T + t
        _, c1 = model(p2)
        expected = c0 @ Q.T + t
        worst = max(worst, float((c1 - expected).abs().max()))
    return worst

w = check_center_equivariance()
print(f"predicted-center equivariance: max error = {w:.3e}")
assert w < 1e-3, "LOCALIZATION HEAD IS NOT SE(3) EQUIVARIANT"
print("holds -- weighted centroid with invariant attention over equivariant positions")

## Cell 5: Training machinery (identical to Phase 14)

In [ ]:
def run_epoch(model, loader, opt=None, lam=0.0):
    tr = opt is not None; model.train(tr)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    huber = nn.SmoothL1Loss(beta=4.0)                    # ~matches the 4A DCC threshold
    ps, ys, cd = [], [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.set_grad_enabled(tr):
            logits, pred_c = model(b)
            loss = crit(logits, b.y)
            if lam > 0 and pred_c is not None:
                true_c = b.true_center.view(-1, 3)
                loss = loss + lam * huber(pred_c, true_c)
                cd.append((pred_c - true_c).norm(dim=1).detach().cpu().numpy())
        if tr:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())
    p, y = np.concatenate(ps), np.concatenate(ys)
    out = {"pr": average_precision_score(y, p), "roc": roc_auc_score(y, p)}
    if cd:
        d_ = np.concatenate(cd)
        out["dcc"] = float((d_ < 4.0).mean()); out["mean_center_err"] = float(d_.mean())
    return out


def fit(use_loc_head, lam, seed, epochs=40, patience=8, batch=16):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LocalizingEGNN(N_PRE, TOPO_DIM, use_loc_head=use_loc_head).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tl = DataLoader(splits["train"], batch_size=batch, shuffle=True)
    vl = DataLoader(splits["val"], batch_size=batch)
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        run_epoch(model, tl, opt, lam)
        v = run_epoch(model, vl, lam=lam)
        sch.step()
        if v["pr"] > best:
            best, state, bad = v["pr"], {k:t.clone() for k,t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); model.eval()
    return model, best

## Cell 6: Collation sanity check before training anything

In [ ]:
# --- verify PyG batches a per-graph (3,) attribute the way this code assumes,
# BEFORE trusting any training run on it. A wrong assumption here would not
# crash -- it would silently train against scrambled targets, which is worse
# than a crash because nothing would look wrong until the results did.
_probe_items = splits["train"][:8]
_expected = torch.stack([d.true_center for d in _probe_items])          # (8,3), known-correct order
_probe_batch = next(iter(DataLoader(_probe_items, batch_size=8)))
_got = _probe_batch.true_center.view(-1, 3)
print(f"true_center collation check: expected {tuple(_expected.shape)}, got {tuple(_got.shape)}")
assert _got.shape == _expected.shape, "COLLATION SHAPE MISMATCH -- true_center is not batching as assumed"
assert torch.allclose(_got, _expected, atol=1e-5), \
    "COLLATION ORDER/VALUE MISMATCH -- b.true_center.view(-1,3) does NOT recover per-graph centers correctly"
print("PASS: b.true_center.view(-1,3) correctly recovers per-graph targets in order")

_probe_loader = DataLoader(splits["train"][:8], batch_size=8)
_model = LocalizingEGNN(N_PRE, TOPO_DIM).to(DEVICE)
_opt = torch.optim.AdamW(_model.parameters(), lr=1e-3)
_r = run_epoch(_model, _probe_loader, _opt, lam=0.1)
print(f"dry run (8 chains, 1 step): {_r}")
assert np.isfinite(_r["pr"]) and "dcc" in _r and np.isfinite(_r["mean_center_err"])
print("gradients flow, losses finite, centroid metric present -- proceeding")

## Cell 7: Train the deployment ensemble (3 seeds, lambda=0.1 from Phase 14's sweep)

`lambda` is not re-tuned here, it was already selected on a held-out val set
in Phase 14 and re-tuning it again on this run's val split would be a second,
unnecessary bite at the same apple.

In [ ]:
BEST_LAMBDA = 0.1   # selected in Phase 14 via val DCC sweep; reused, not re-tuned

WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)
ensemble = []
t0 = time.time()
for seed in (0,1,2):
    m, val_pr = fit(use_loc_head=True, lam=BEST_LAMBDA, seed=seed)
    v = run_epoch(m, DataLoader(splits["val"], batch_size=16), lam=BEST_LAMBDA)
    ensemble.append(m)
    torch.save(m.state_dict(), WORK/f"cache/loc_deploy_seed{seed}.pt")
    val_dcc = v.get("dcc", float("nan"))
    print(f"  seed {seed}: val PR {val_pr:.4f}  val DCC {val_dcc:.4f}  "
          f"[{(time.time()-t0)/60:.1f}m]")
print(f"deployment ensemble of {len(ensemble)} trained on {len(dataset):,} chains")

## Cell 8: Fetch benchmark structures (identical to Phase 12)

In [ ]:
def parse_ds(url, prefix):
    lines = urllib.request.urlopen(url, timeout=60).read().decode().splitlines()
    out = []
    for ln in lines:
        ln = ln.strip()
        if not ln or ln.startswith("#"): continue
        name = ln.split("/")[-1].replace(".pdb", "")
        if prefix == "coach420":
            out.append((name[:4].upper(), name[4:].upper() or None))   # (pdb_id, chain or None)
        else:
            out.append((name[:4].upper(), None))
    return out

coach = parse_ds("https://raw.githubusercontent.com/rdk/p2rank-datasets/master/coach420.ds", "coach420")
holo  = parse_ds("https://raw.githubusercontent.com/rdk/p2rank-datasets/master/holo4k.ds", "holo4k")
print(f"COACH420: {len(coach)} entries | HOLO4K: {len(holo)} entries")

bench = [("coach420", pid, ch) for pid, ch in coach] + [("holo4k", pid, ch) for pid, ch in holo]
need = sorted({pid for _, pid, _ in bench})
print(f"unique PDB IDs to fetch: {len(need):,}")

def fetch(pid):
    p = RAW / f"{pid}.pdb.gz"
    if p.exists() and p.stat().st_size > 500: return pid, p
    try:
        with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.pdb.gz", timeout=60) as r:
            b = r.read()
        if len(b) > 500: p.write_bytes(b); return pid, p
    except Exception: pass
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=24) as ex: got = list(ex.map(fetch, need))
paths = {p: f for p, f in got if f}
print(f"fetched {len(paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

## Cell 9: Ground-truth ligand extraction + leakage check (identical to Phase 12)

In [ ]:
from Bio.PDB import PDBParser

AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}
NON_LIGAND={"HOH","DOD","WAT","SO4","PO4","GOL","EDO","PEG","PG4","PGE","1PE","MPD","TRS",
            "ACT","ACY","FMT","CIT","MES","EPE","DMS","IMD","BME","NO3","AZI","CO3","IOD",
            "BR","CL","F","NA","K","CS","RB","LI","UNX","UNL","UNK"}
CATALYTIC_METALS={"ZN","MG","MN","FE","FE2","CU","CU1","CA","NI","CO","CD","W","MO"}
MIN_RES, MAX_RES, MIN_LIG_ATOMS = 50, 1000, 6

train_pdb_ids = {d.pdb_id for d in train_dataset}
overlap_exact = {pid for _, pid, _ in bench if pid in train_pdb_ids}
print(f"exact PDB-ID overlap with training set: {len(overlap_exact)}/{len(need)} unique IDs")

def parse_structure(pid, path, wanted_chain):
    """Returns list of (chain_id, seq, ca_xyz, per_res_atoms, ligand_list)."""
    try:
        with gzip.open(path, "rt") as fh:
            model = next(PDBParser(QUIET=True, PERMISSIVE=True).get_structure(pid, fh).get_models())
    except Exception:
        return []
    ligs = []
    for ch in model:
        for res in ch:
            if res.id[0] == " ": continue
            nm = res.get_resname().strip().upper()
            if nm in AA3_TO_1 or nm in NON_LIGAND: continue
            xyz = np.asarray([a.coord for a in res.get_unpacked_list() if a.element not in ("H","D")],
                            dtype=np.float32).reshape(-1,3)
            is_metal = nm in CATALYTIC_METALS
            if is_metal or len(xyz) >= MIN_LIG_ATOMS:
                ligs.append(xyz)
    out = []
    chains = [c for c in model] if wanted_chain is None else \
             [model[wanted_chain]] if wanted_chain in [c.id for c in model] else []
    for ch in chains:
        residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
        if not (MIN_RES <= len(residues) <= MAX_RES): continue
        ca=[r["CA"].coord for r in residues]
        seq="".join(AA3_TO_1.get(r.get_resname().strip().upper(),"X") for r in residues)
        atoms=[np.asarray([a.coord for a in r.get_unpacked_list() if a.element not in ("H","D")],
                          dtype=np.float32).reshape(-1,3) for r in residues]
        out.append((ch.id, seq, np.asarray(ca,dtype=np.float32), atoms, ligs))
    return out

t0=time.time()
parsed = {}   # pdb_id -> list of chain tuples, ALL chains parsed once per structure
for pid in sorted(paths):
    parsed[pid] = parse_structure(pid, paths[pid], None)
print(f"parsed {len(parsed)} structures in {(time.time()-t0)/60:.1f} min")

# assemble the evaluation unit list: one entry per (benchmark, pdb_id, target_chain_or_None)
eval_units = []
for src, pid, ch in bench:
    chains = parsed.get(pid, [])
    if not chains: continue
    if src == "coach420":
        match = [c for c in chains if c[0] == ch]
        if match: eval_units.append((src, pid, [match[0]]))
    else:
        # HOLO4K evaluates the whole assembly, but some entries are large
        # multi-chain complexes (viral capsids etc.) -- cap to the 6 longest
        # chains so one pathological structure cannot dominate runtime. This
        # covers the large majority of biologically relevant subunits; the
        # count actually capped is reported below rather than silently dropped.
        if chains:
            capped = sorted(chains, key=lambda c: -len(c[1]))[:6]
            eval_units.append((src, pid, capped))
n_capped = sum(1 for src,pid,ch in eval_units if src=="holo4k"
              and len(parsed.get(pid,[])) > len(ch))
print(f"evaluable units (structure parsed, target chain present): {len(eval_units)}/{len(bench)}")
print(f"HOLO4K structures with >6 chains, capped to 6 longest: {n_capped}")

In [ ]:
# --- MMseqs2 homology check against the training set --------------------------
mm = None
try:
    subprocess.run("curl -sL https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz -o /tmp/mm.tar.gz "
                   "&& tar xzf /tmp/mm.tar.gz -C /tmp", shell=True, check=True, timeout=600)
    c = Path("/tmp/mmseqs/bin/mmseqs")
    if c.exists(): os.chmod(c, 0o755); mm = str(c)
except Exception as e:
    print("mmseqs2 unavailable:", type(e).__name__)

W = Path("/tmp/leak_mm"); shutil.rmtree(W, ignore_errors=True); W.mkdir()
fasta_lines = []
for d in train_dataset:
    fasta_lines.append(f">TRAIN__{d.pdb_id}_{d.chain_id}\n{d.seq}\n")
bench_seq_map = {}
for src, pid, chains in eval_units:
    for cid, seq, *_ in chains:
        key = f"BENCH__{src}__{pid}_{cid}"
        bench_seq_map[key] = (src, pid, cid)
        fasta_lines.append(f">{key}\n{seq}\n")
(W/"all.fasta").write_text("".join(fasta_lines))

subprocess.run([mm, "easy-cluster", str(W/"all.fasta"), str(W/"res"), str(W/"tmp"),
                "--min-seq-id", "0.30", "-c", "0.8", "--cov-mode", "1", "-v", "1"],
               check=True, capture_output=True, timeout=3600)
rep = defaultdict(list)
for line in (W/"res_cluster.tsv").read_text().splitlines():
    a, b = line.split("\t"); rep[a].append(b)
clusters = list(rep.values()) if rep else []
# merge singleton reps (mmseqs always includes the rep itself in its own cluster list)
for r_, members in rep.items():
    if r_ not in members: members.append(r_)

leaked_keys = set()
for members in rep.values():
    has_train = any(m.startswith("TRAIN__") for m in members)
    has_bench = any(m.startswith("BENCH__") for m in members)
    if has_train and has_bench:
        leaked_keys.update(m for m in members if m.startswith("BENCH__"))

print(f"benchmark chains flagged as homologous (>=30% identity) to a training chain: "
      f"{len(leaked_keys)}/{len(bench_seq_map)}")

clean_units = []
for src, pid, chains in eval_units:
    clean_chains = [c for c in chains if f"BENCH__{src}__{pid}_{c[0]}" not in leaked_keys
                    and pid not in overlap_exact]
    if clean_chains:
        clean_units.append((src, pid, clean_chains))
n_dropped = len(eval_units) - len(clean_units)
print(f"\nclean (non-leaked) evaluation units: {len(clean_units)}/{len(eval_units)} "
      f"({n_dropped} dropped to leakage)")
for name in ("coach420","holo4k"):
    tot = sum(1 for s,_,_ in eval_units if s==name)
    cln = sum(1 for s,_,_ in clean_units if s==name)
    print(f"  {name:<10} {cln}/{tot} clean")

## Cell 10: Featurize the clean benchmark chains, AND cache raw features this time

Phase 12 only cached derived outputs (probabilities); Phase 13 needed to redo
the ~170-minute featurization to test new heuristics. This time the raw
per-chain tensors are saved directly, so any future architecture iteration on
this same benchmark set is free.

In [ ]:
import gudhi
from transformers import AutoTokenizer, AutoModel
from Bio.PDB import PPBuilder

AA_ALPHABET="ACDEFGHIKLMNPQRSTVWYX"; A2I={a:i for i,a in enumerate(AA_ALPHABET)}
MAX_HOM_DIM=2; BETTI_RADII=(2.0,3.0,5.0); NBHD_RADIUS=12.0; MAX_NBHD_ATOMS=300
FPD=6+2+len(BETTI_RADII)

# Phase 6b pruned one constant column (35 raw features -> 34) before the model
# ever saw them. Reconstruct the full 35-name list with the SAME generation
# logic used in every prior phase, then match by NAME against blob6b's saved
# 34-name list to recover which column was dropped -- matching by name (not a
# hardcoded index) means this self-verifies via the assert rather than silently
# drifting if the pruned column ever changes.
_FULL_TOPO_NAMES = [f"H{d}_{s}" for d in range(MAX_HOM_DIM+1)
                    for s in ["n_bars","total_pers","max_pers","mean_pers","std_pers","entropy",
                              "mean_birth","mean_death", *[f"betti_r{r:g}" for r in BETTI_RADII]]]                    + ["nbhd_n_atoms","nbhd_gyration"]
_TOPO_KEEP = np.array([n in set(TOPO_NAMES) for n in _FULL_TOPO_NAMES])
assert _TOPO_KEEP.sum() == TOPO_DIM,     f"expected {TOPO_DIM} surviving topo columns, matched {_TOPO_KEEP.sum()} by name"
print(f"topo pruning reconstructed: {len(_FULL_TOPO_NAMES)} raw -> {_TOPO_KEEP.sum()} kept "
      f"(dropped: {[n for n,k in zip(_FULL_TOPO_NAMES,_TOPO_KEEP) if not k]})")

def rgc(pos,r=8.0,k=32):
    n=pos.size(0); pos=pos-pos.mean(0,keepdim=True)
    d=torch.cdist(pos,pos).clone(); d.fill_diagonal_(float("inf")); m=d<=r
    if k<n:
        _,i=torch.topk(d,k,dim=1,largest=False); keep=torch.zeros_like(m); keep.scatter_(1,i,True); m&=keep
    m=m|m.t(); t,s=m.nonzero(as_tuple=True); return torch.stack([s,t]).long()

def _dg(pts):
    pts=np.ascontiguousarray(pts,dtype=np.float64); st=None; sq=True
    if len(pts)>=5:
        try: st=gudhi.AlphaComplex(points=pts).create_simplex_tree()
        except Exception: st=None
    if st is None:
        st=gudhi.RipsComplex(points=pts,max_edge_length=float(NBHD_RADIUS)).create_simplex_tree(max_dimension=MAX_HOM_DIM+1); sq=False
    st.compute_persistence(homology_coeff_field=2,persistence_dim_max=True)
    o={}
    for d in range(MAX_HOM_DIM+1):
        b=np.asarray(st.persistence_intervals_in_dimension(d),dtype=np.float64).reshape(-1,2)
        o[d]=np.sqrt(np.maximum(b,0.0)) if (sq and len(b)) else b
    return o

def _vec(dg,pts):
    f=[]
    for d in range(MAX_HOM_DIM+1):
        bars=dg.get(d,np.zeros((0,2)))
        fin=bars[np.isfinite(bars).all(axis=1)] if len(bars) else bars.reshape(-1,2)
        if len(fin)==0: f.extend([0.0]*FPD); continue
        b,dt=fin[:,0],fin[:,1]; pers=np.maximum(dt-b,0.0); tot=float(pers.sum())
        ent=0.0
        if tot>0:
            p=pers/tot; nz=p[p>0]; ent=float(-(nz*np.log(nz)).sum())
        f.extend([float(len(fin)),tot,float(pers.max()),float(pers.mean()),float(pers.std()),
                  ent,float(b.mean()),float(dt.mean()),
                  *[float(((b<=r)&(dt>r)).sum()) for r in BETTI_RADII]])
    c=pts-pts.mean(0,keepdims=True)
    f.extend([float(len(pts)),float(np.sqrt((c**2).sum(1).mean())) if len(pts) else 0.0])
    return np.asarray(f,dtype=np.float32)

def topo_chain(ca,per_res):
    atoms=np.concatenate([a for a in per_res if len(a)],axis=0)
    tree=cKDTree(atoms); out=np.zeros((len(ca),TOPO_DIM),np.float32)
    for i,c in enumerate(ca):
        idx=tree.query_ball_point(c,NBHD_RADIUS)
        if len(idx)>MAX_NBHD_ATOMS: _,idx=tree.query(c,k=MAX_NBHD_ATOMS)
        pts=atoms[np.asarray(idx,dtype=int)]
        if len(pts)>=4: out[i]=_vec(_dg(pts),pts)[_TOPO_KEEP]
    return out

def dihedrals(pdb_id, chain_id, path, n_expect):
    try:
        with gzip.open(path,"rt") as fh:
            model=next(PDBParser(QUIET=True,PERMISSIVE=True).get_structure(pdb_id,fh).get_models())
        ch=model[chain_id]
    except Exception: return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
    if len(residues)!=n_expect: return None
    ppmap={}
    for pp in PPBuilder().build_peptides(ch, aa_only=False):
        for res,(phi,psi) in zip(pp, pp.get_phi_psi_list()): ppmap[res.id]=(phi,psi)
    phis,psis=[],[]
    for r in residues:
        phi,psi=ppmap.get(r.id,(None,None)); phis.append(phi); psis.append(psi)
    def enc(a):
        v=torch.tensor([x if x is not None else 0.0 for x in a],dtype=torch.float32)
        m=torch.tensor([x is not None for x in a],dtype=torch.float32)
        return torch.stack([torch.sin(v)*m,torch.cos(v)*m],1), m.unsqueeze(1)
    pf,pm=enc(phis); sf,sm=enc(psis)
    return torch.cat([pf,sf,pm,sm],1)

tok = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")
esm = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D").to(DEVICE).eval()

@torch.no_grad()
def embed_seq(seq):
    enc = tok([seq], return_tensors="pt", truncation=True, max_length=1002)
    enc = {k:v.to(DEVICE) for k,v in enc.items()}
    out = esm(**enc).last_hidden_state
    return out[0,1:1+len(seq),:].float().cpu()

t0=time.time(); built = {}   # key -> Data-like dict
n_chains_total = sum(len(c) for _,_,c in clean_units)
done=0
for src, pid, chains in clean_units:
    for cid, seq, ca, per_res, ligs in chains:
        n = len(ca)
        oh = torch.zeros(n, 21); oh[torch.arange(n), torch.tensor([A2I.get(c,20) for c in seq])]=1.0
        dih = dihedrals(pid, cid, paths[pid], n)
        if dih is None: continue
        esm_e = embed_seq(seq)
        topo = topo_chain(ca, per_res)
        pos = torch.from_numpy(ca); pos = pos - pos.mean(0, keepdim=True)
        x = torch.cat([oh, dih, esm_e, torch.from_numpy(topo.astype(np.float32))], dim=1)
        ei = rgc(pos)
        d_ = (pos[ei[0]]-pos[ei[1]]).norm(dim=1,keepdim=True)
        sep = (ei[0]-ei[1]).abs().float().unsqueeze(1)
        built[(src,pid,cid)] = dict(x=x, pos=pos, edge_index=ei,
                                    edge_attr=torch.cat([d_,torch.log1p(sep)],1),
                                    ca=ca, ligs=ligs)
        done += 1
    if done % 100 < len(chains): print(f"  {done}/{n_chains_total}  [{(time.time()-t0)/60:.1f}m]")

del esm; torch.cuda.empty_cache()
print(f"\nfeaturized {len(built)}/{n_chains_total} clean chains in {(time.time()-t0)/60:.1f} min")
torch.save({"built_keys": list(built.keys())}, WORK/"cache/built_index.pt")

In [ ]:
# Cache raw features (x, pos, edge_index, edge_attr) -- NOT just derived probs --
# so a future model change can be evaluated without repeating ~170 min of
# fetch+parse+ESM+topology.
import pickle as _pkl
raw_cache = {k: {kk: (vv.numpy() if torch.is_tensor(vv) else vv) for kk,vv in v.items()}
            for k,v in built.items()}
with open(WORK/"cache/phase15_raw_features.pkl", "wb") as fh:
    _pkl.dump(raw_cache, fh)
print(f"cached raw features for {len(raw_cache)} chains -> "
      f"{(WORK/'cache/phase15_raw_features.pkl').stat().st_size/1e6:.0f} MB")

## Cell 11: Inference with the trained localization head, no heuristic

In [ ]:
from collections import defaultdict

def dcc_dca(pred_center, ligs):
    if pred_center is None or not ligs: return None, None
    best_dcc, best_dca = np.inf, np.inf
    for lig_atoms in ligs:
        if len(lig_atoms) == 0: continue
        com = lig_atoms.mean(axis=0)
        best_dcc = min(best_dcc, float(np.linalg.norm(pred_center - com)))
        best_dca = min(best_dca, float(np.linalg.norm(lig_atoms - pred_center, axis=1).min()))
    return (best_dcc if np.isfinite(best_dcc) else None,
            best_dca if np.isfinite(best_dca) else None)


mu, sd = blob6b["topo_mean"], blob6b["topo_std"]

@torch.no_grad()
def predict_chain(entry):
    x = entry["x"].clone()
    x[:, N_PRE:] = (x[:, N_PRE:] - torch.from_numpy(mu)) / torch.from_numpy(sd)
    d = type("D", (), {})()
    d.x=x.to(DEVICE); d.pos=entry["pos"].to(DEVICE)
    d.edge_index=entry["edge_index"].to(DEVICE); d.edge_attr=entry["edge_attr"].to(DEVICE)
    d.batch = torch.zeros(x.size(0), dtype=torch.long, device=DEVICE)
    seed_logits, seed_centers, seed_offsets = [], [], []
    for m in ensemble:
        lg, pc = m(d)
        seed_logits.append(torch.sigmoid(lg).cpu().numpy())
        seed_centers.append(pc.cpu().numpy()[0])                 # (3,) -- single graph
    probs = np.mean(seed_logits, axis=0)
    center = np.mean(seed_centers, axis=0)                       # ensemble-average predicted point
    return probs, center


t0 = time.time()
by_structure = defaultdict(list)
for (src_,pid,cid), entry in built.items():
    by_structure[(src_,pid)].append((cid, entry))

predictions = {}
for (src_, pid), chain_list in by_structure.items():
    ligs = chain_list[0][1]["ligs"]
    best_conf, best_center = -np.inf, None
    for cid, entry in chain_list:
        probs, center = predict_chain(entry)
        conf = float(probs.sum())                                # per-chain "pocket-likelihood"
        if conf > best_conf:
            best_conf, best_center = conf, center
    predictions[(src_,pid)] = (best_center, ligs)

print(f"predicted centers for {len(predictions)} structures in {(time.time()-t0)/60:.1f} min "
      f"(own-centroid, no clustering heuristic)")

## Cell 12: Score, compare against Phase 12/13 and P2Rank

In [ ]:
PHASE12_REFERENCE = {
    "coach420": {"dcc": 0.3678, "dca": 0.6782},
    "holo4k":   {"dcc": 0.4033, "dca": 0.7053},
}
P2RANK_REFERENCE = {
    "coach420": {"dcc": 0.4706, "dca": 0.6941},
    "holo4k":   {"dcc": 0.5509, "dca": 0.7796},
}
PHASE13_POOLED_TUNED_DCC = 0.4355   # pooled coach420+holo4k, different tune/test split

results = {"coach420": [], "holo4k": []}
for (src_, pid), (center, ligs) in predictions.items():
    dcc, dca = dcc_dca(center, ligs)
    if dcc is not None: results[src_].append((dcc, dca))

print(f"{'benchmark':<10}{'n':>6}{'DCC succ':>10}{'DCA succ':>10}")
print("-"*36)
summary = {}
all_dcc = []
for src_ in ("coach420","holo4k"):
    vals = results[src_]
    if not vals: print(f"{src_:<10}{0:>6}{'--':>10}{'--':>10}"); continue
    dcc_s = np.mean([1.0 if d<4.0 else 0.0 for d,_ in vals])
    dca_s = np.mean([1.0 if a<4.0 else 0.0 for _,a in vals])
    summary[src_] = {"n": len(vals), "dcc": float(dcc_s), "dca": float(dca_s)}
    all_dcc.extend([1.0 if d<4.0 else 0.0 for d,_ in vals])
    print(f"{src_:<10}{len(vals):>6}{dcc_s:>10.3f}{dca_s:>10.3f}")

pooled_dcc = float(np.mean(all_dcc)) if all_dcc else float("nan")

print(f"\n{'':30}{'DCC':>8}")
print("-"*40)
for src_ in ("coach420","holo4k"):
    if src_ not in summary: continue
    p12v = PHASE12_REFERENCE[src_]["dcc"]; p2v = P2RANK_REFERENCE[src_]["dcc"]
    ours = summary[src_]["dcc"]
    print(f"  {src_:<10} ours {ours:.4f}  |  Phase12(heuristic) {p12v:.4f}  ({ours-p12v:+.4f})  |  "
          f"P2Rank {p2v:.4f}  ({ours-p2v:+.4f})")

print(f"\n  pooled DCC (both benchmarks): {pooled_dcc:.4f}  "
      f"vs Phase 13 tuned-heuristic pooled: {PHASE13_POOLED_TUNED_DCC:.4f}  "
      f"({pooled_dcc-PHASE13_POOLED_TUNED_DCC:+.4f})")

json.dump({"per_benchmark": summary, "pooled_dcc": pooled_dcc,
           "phase12_reference": PHASE12_REFERENCE, "p2rank_reference": P2RANK_REFERENCE,
           "phase13_pooled_tuned_dcc": PHASE13_POOLED_TUNED_DCC},
          open(WORK/"phase15_summary.json", "w"), indent=2)
print("\nsaved -> phase15_summary.json")

---

## Reading this

Three comparisons, in order of how much they can be trusted:

1. **vs Phase 12's classification-only model** (same benchmark, same leakage
   filter, same scoring code, only the model differs), the cleanest
   before/after in this notebook. A clear win here means the localization
   training signal generalizes beyond our own held-out split to a genuinely
   external, harder benchmark.
2. **vs P2Rank** (same-run, same clean subset, controlled), the number that
   answers "is this competitive with the field's standard tool", the same
   question Phase 12 first asked and answered "no" to.
3. **vs Phase 13's pooled tuned-heuristic number**, informative but on a
   different tune/test split of the same clean set, so treat this one as
   directional.

If this closes most or all of the P2Rank gap, that is a real, externally
validated result: a trained localization objective beats both a classification
-only model and a purpose-tuned post-hoc heuristic, on the field's own
benchmark. If it falls short of Phase 12's baseline despite winning cleanly on
our own held-out split in Phase 14, that is itself informative, it would mean
the auxiliary loss overfit to the training distribution's ligand-placement
patterns in a way that doesn't transfer, which is a different and equally
useful thing to know before calling this approach finished.